# Day 53 — MLOps intro: experiment tracking with MLflow
Objectives:
- Track params, metrics, and artifacts with MLflow.
- Organize experiments and runs.
- Save and load models from MLflow.
Note: `pip install mlflow` (already in requirements.txt). By default this uses a local `mlruns` folder.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
X,y = load_breast_cancer(return_X_y=True)
Xtr,Xte,ytr,yte = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)
pipe = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])
mlflow.set_experiment('ds-60day-bc-experiment')
with mlflow.start_run(run_name='baseline-logreg'):
    mlflow.log_param('model', 'LogisticRegression')
    mlflow.log_param('scale', True)
    pipe.fit(Xtr,ytr)
    yprob = pipe.predict_proba(Xte)[:,1]
    auc = roc_auc_score(yte, yprob)
    mlflow.log_metric('roc_auc', auc)
    mlflow.sklearn.log_model(pipe, artifact_path='model')
    print('ROC AUC:', auc)


## Viewing results
Run the MLflow UI in a terminal:
```bash
mlflow ui --backend-store-uri mlruns
```
Then open http://127.0.0.1:5000 to view experiments.

## Loading a model from MLflow
You can load the saved model artifact and use it for inference.

In [ ]:
# Example: load the last logged model (adjust run_id/artifact URI as needed)
# model_uri = 'runs:/<run_id>/model'
# loaded = mlflow.sklearn.load_model(model_uri)
# loaded.predict(Xte[:5])


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — MLflow run identity, params, metrics, artifacts, and reproducible evidence

### Mental model

Experiment tracking records evidence about a run; it does not make the
run reproducible by itself. An experiment groups comparable runs. A run
represents one execution. Parameters describe configuration, metrics
are numeric observations (possibly by step), tags add searchable
context, and artifacts preserve files or models.

Comparability requires the same data/split/metric definitions. A model
artifact should carry an input example/signature and provenance. Local
file tracking is useful for study but is not a shared registry, access
control system, or backup.

### Read the API before running it

- **`with mlflow.start_run():`:** creates a bounded lifecycle so successful and failed runs receive terminal status.
- **`log_param` versus `log_metric`:** stores fixed configuration separately from numeric measurements that may evolve by step.
- **`log_artifact` / model logging:** copies output into the run's artifact store; verify reload and input contract rather than trusting existence.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — separate comparable configuration from results

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** The data snapshot and metric definition are stable enough that two records are actually comparable.

In [ ]:
run_record = {
    "params": {
        "model": "logistic_regression",
        "C": 1.0,
        "split_seed": 5301,
        "data_snapshot": "sha256:example",
    },
    "metrics": {
        "validation_roc_auc": 0.91,
        "test_roc_auc": 0.89,
    },
    "tags": {
        "purpose": "course-baseline",
        "metric_definition": "roc_auc",
    },
}
assert "validation_roc_auc" not in run_record["params"]
print(run_record)

**Expected observation:** Configuration, measured outcomes, and contextual labels are distinct, making search and comparison less ambiguous.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — create and inspect a fully local temporary run

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** The temporary store proves API mechanics only; it is intentionally deleted and not a durable team record.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import mlflow

original_uri = mlflow.get_tracking_uri()
with TemporaryDirectory() as directory:
    mlflow.set_tracking_uri(Path(directory).as_uri())
    mlflow.set_experiment("day53-local-mechanics")
    with mlflow.start_run(run_name="bounded-example") as active:
        mlflow.log_param("alpha", 0.1)
        mlflow.log_metric("validation_score", 0.8)
        run_id = active.info.run_id
    recorded = mlflow.get_run(run_id)
    print(recorded.info.status, recorded.data.params, recorded.data.metrics)
    assert recorded.info.status == "FINISHED"
mlflow.set_tracking_uri(original_uri)

**Expected observation:** The context manager closes the run as FINISHED, and the parameter/metric are queryable from a local temporary tracking store.

### Debugging and practice ramp

**Common mistake:** Comparing runs with different splits/metrics or logging only the model score while omitting data and environment identity.

**Diagnostic:** Query the run by ID, inspect status/params/metrics/tags/artifacts, reload the model in a fresh process, and reconcile inputs and predictions.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define MLflow run identity, params, metrics, artifacts, and reproducible evidence in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not promote the top UI row without acceptance gates, comparable evidence, artifact verification, and ownership.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Log additional parameters such as Logistic Regression `C` and compare runs.

**Verify:** For task `Log additional parameters such as Logistic Regression C and compare runs`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed; then record the exact command/input, terminal result or returned value, and repeat the critical check from a clean process or fresh state.






2. Save a confusion-matrix PNG and log it as an artifact.

**Verify:** For task `Save a confusion-matrix PNG and log it as an artifact`, verify identity/hash and metadata, then reload or inspect the artifact outside the creating state and test one tampered mismatch.






3. Try a different classifier, such as Random Forest, and compare ROC AUC.

**Verify:** For task `Try a different classifier, such as Random Forest, and compare ROC AUC`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.







### Progressive hints

1. Make one run per configuration and include split seed, metric name, and model
   type. Avoid changing several uncontrolled factors at once.
2. Save figures under an ignored `artifacts/` directory, close the figure, and
   pass the path to `mlflow.log_artifact`.
3. Reuse exactly the same train/test split. Compare runtime and complexity as
   well as score.

The reference solution adds scikit-learn autologging and model reload. Start
with explicit logging so you know which information is essential; use autolog
as a supplement, not as a substitute for experiment design.

### Additional mastery practice

Make experiment records reconstructable: status, parameters, data/code identity, metrics, artifacts, and model signature must describe one coherent run.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Failure-state handling:** Run an experiment that intentionally raises after logging parameters. Verify MLflow records a failed status and useful exception context without exposing raw data or secrets.
   **Progressive hint:** Use the run context manager so exception exit marks the run failed. Log safe stage/status information before re-raising.

**Verify:** For task `Failure-state handling: Run an experiment that intentionally raises after logging parameters....`, reproduce the failure first, capture its smallest observable symptom, apply one scoped fix, and rerun the failing plus normal case; then record the exact command/input, terminal result or returned value, and repeat the critical check from a clean process or fresh state.







5. **Provenance manifest:** Log a JSON provenance artifact containing data fingerprint, code revision, dependency lock hash, feature schema, split policy, and metric definitions.
   **Progressive hint:** Use portable identifiers and hashes, not developer-specific absolute paths. Validate required fields before ending the run.

**Verify:** For task `Provenance manifest: Log a JSON provenance artifact containing data fingerprint, code revisio...`, record the exact command/input, terminal result or returned value, and repeat the critical check from a clean process or fresh state; then assert exact names, order, types/nullability or versions and prove one mismatch is rejected rather than silently coerced.







6. **Reload and signature check:** Log a fitted pipeline with an input example/signature, reload it by run URI, and assert prediction parity on a fixed fixture.
   **Progressive hint:** The fixture must use the documented schema and never come from hidden notebook state. Compare probabilities within a tolerance.

**Verify:** For task `Reload and signature check: Log a fitted pipeline with an input example/signature, reload it...`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.






Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Failure-state handling


# Practice 5 — Provenance manifest


# Practice 6 — Reload and signature check
